# 第42章 抖动散点图（stripplot）

用轻微抖动展示分类组中的每一个原始观察。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

样本量中小，需要保留真实点并查看重叠和离群。

## 数据结构

分类变量与数值变量，每行一条观察。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 jitter 参数从 0.22 改为 0.05 或 0.4，观察抖动幅度对点分散程度的影响
2. 调整 alpha 参数（如 0.3 或 0.9），说明透明度对重叠点可见性的作用
3. 修改 dodge=True 为 dodge=False，对比分组错位与叠加显示的视觉效果


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="notebook")
from js import window
base_url = window.location.origin
diamonds = pd.read_csv(f"{base_url}/datasets/diamonds.csv")
orders_full = diamonds.assign(
    category=diamonds["cut"], channel=diamonds["color"], region=diamonds["clarity"],
    order_value=diamonds["price"], items=diamonds["carat"],
    satisfied=np.where(diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"),
)
orders = orders_full.sample(2_000, random_state=36).copy()
taxis = pd.read_csv(f"{base_url}/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"), visits=taxis["distance"],
    ad_spend=taxis["tip"], sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(min(2_000, len(marketing_full)), random_state=36).copy()
flights = pd.read_csv(f"{base_url}/datasets/flights.csv")
daily = flights.assign(
    date=pd.to_datetime(flights["year"].astype("string") + "-" + flights["month"] + "-01"),
    region="AirPassengers", sales=flights["passengers"],
)
print(f"Diamonds：{len(diamonds):,} 行；NYC Taxis：{len(taxis):,} 行；Flights：{len(flights):,} 行")
print("图表兼容列均由公开数据原始字段直接映射；高成本图使用固定 2,000 行样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
sample = orders.sample(120, random_state=42)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.stripplot(data=sample, x="category", y="order_value", jitter=0.22, alpha=0.55, color="#1a73e8", ax=ax)
ax.set(title="品类客单价原始观察", xlabel="品类", ylabel="客单价（元）")
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
sample = orders.sample(150, random_state=420)
fig, ax = plt.subplots(figsize=(9, 4.8))
sns.stripplot(data=sample, x="category", y="order_value", hue="channel", dodge=True, jitter=0.18, alpha=0.6, palette="colorblind", ax=ax)
ax.set(title="分渠道展示原始订单", xlabel="品类", ylabel="客单价（元）")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()


## 3. 参数说明

- jitter：水平抖动
- size：点大小
- alpha：透明度
- dodge：hue错位


## 4. 结果解读

读取点密度、范围和异常值；抖动只改变显示位置，不改变数据。


## 常见误区

- 点太多造成黑块
- 抖动过大导致类别边界模糊
- 不透明点遮挡重叠


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
sample = orders.sample(100, random_state=421)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=sample, x="region", y="order_value", color="white", showfliers=False, ax=ax)
sns.stripplot(data=sample, x="region", y="order_value", color="#188038", alpha=0.5, jitter=0.2, ax=ax)
ax.set(title="箱线摘要与原始点", xlabel="区域", ylabel="客单价（元）")
fig.tight_layout()
plt.show()


## 本章小结

用轻微抖动展示分类组中的每一个原始观察。


### 你已经掌握

- 判断抖动散点图（stripplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 样本量中小，需要保留真实点并查看重叠和离群。 |
| 数据结构 | 分类变量与数值变量，每行一条观察。 |
| 结果解读 | 读取点密度、范围和异常值；抖动只改变显示位置，不改变数据。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `jitter` | 水平抖动 |
| `size` | 点大小 |
| `alpha` | 透明度 |
| `dodge` | hue错位 |


### 需要注意

- 点太多造成黑块
- 抖动过大导致类别边界模糊
- 不透明点遮挡重叠


### 完成检查

- [ ] 能判断什么问题适合使用抖动散点图（stripplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
